In [27]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

# This searches for a .env file and loads the variables
load_dotenv() 

# Now you can access them using os.getenv
api_key = os.getenv("GOOGLE_API_KEY")

In [28]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
    
)

In [29]:
llm.invoke("Hi").content

'Hi there! How can I help you today?'

In [30]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal, Optional,Annotated
from typing import TypedDict, Optional, List
import pandas as pd

In [31]:
import pandas as pd
import numpy as np


def generate_fake_data(n_wafers=50, seed=42):
    np.random.seed(seed)
    
    wafer_ids = [f"W{i:03d}" for i in range(n_wafers)]
    
    # --- Base Process ---
    base_mean = 100
    base_sigma = 2
    
    # Current Layer
    current_mean = np.random.normal(base_mean, 1.5, n_wafers)
    current_sigma = np.random.normal(base_sigma, 0.3, n_wafers)
    
    UCL = np.full(n_wafers, base_mean + 3 * base_sigma)
    LCL = np.full(n_wafers, base_mean - 3 * base_sigma)
    
    current_df = pd.DataFrame({
        "wafer_id": wafer_ids,
        "mean": current_mean,
        "sigma": current_sigma,
        "UCL": UCL,
        "LCL": LCL,
        "target": base_mean
    })
    
    # --- Inject anomalies (important!) ---
    anomaly_idx = np.random.choice(n_wafers, size=8, replace=False)
    
    # Some wafers above UCL
    current_df.loc[anomaly_idx[:3], "mean"] += 8
    
    # Some below LCL
    current_df.loc[anomaly_idx[3:5], "mean"] -= 8
    
    # Some high sigma
    current_df.loc[anomaly_idx[5:], "sigma"] += 2
    
    
    # --- Previous Layer ---
    prev_mean = current_mean + np.random.normal(0, 1, n_wafers)
    prev_sigma = current_sigma + np.random.normal(0, 0.2, n_wafers)
    
    prev_df = pd.DataFrame({
        "wafer_id": wafer_ids,
        "mean": prev_mean,
        "sigma": prev_sigma
    })
    
    # Make some anomalies come from previous layer
    prev_df.loc[anomaly_idx[:4], "mean"] = current_df.loc[anomaly_idx[:4], "mean"]
    
    
    # --- Qualification Data ---
    thk_mean = np.random.normal(50, 2, n_wafers)
    thk_sigma = np.random.normal(1.5, 0.2, n_wafers)
    
    qual_df = pd.DataFrame({
        "wafer_id": wafer_ids,
        "thk_mean": thk_mean,
        "thk_sigma": thk_sigma,
        "thk_target": 50
    })
    
    # Inject thickness issue for some wafers
    qual_df.loc[anomaly_idx[4:], "thk_mean"] -= 5
    
    return current_df, prev_df, qual_df

In [32]:
current_df, prev_df, qual_df = generate_fake_data()

In [33]:
from typing import TypedDict, Dict
import pandas as pd

class ProcessState(TypedDict, total=False):
    current_data: pd.DataFrame
    prev_data: pd.DataFrame
    qual_data: pd.DataFrame

    anomalies: pd.DataFrame
    comparison_results: pd.DataFrame
    root_cause: pd.DataFrame

    summaries: Dict[str, str]

In [34]:
def gemini_summarize(title: str, df: pd.DataFrame) -> str:
    if df is None or df.empty:
        return f"{title}: No data available"

    prompt = f"""
    You are a semiconductor process engineer.

    Step: {title}

    Analyze this wafer data and give short technical insights:

    {df.head(20).to_string(index=False)}

    Keep it concise.
    """
    
    response = llm.invoke(prompt)
    return response.content

In [35]:
def compute_sigma(state: ProcessState):
    df = state["current_data"].copy()
    
    df["upper_1sigma"] = df["mean"] + df["sigma"]
    df["lower_1sigma"] = df["mean"] - df["sigma"]
    
    summary = gemini_summarize(
        f"Summarize sigma bounds calculation:\n{df.head().to_string()}"
    )
    
    state["current_data"] = df
    state.setdefault("summaries", {})["sigma"] = summary
    return state

In [36]:
def detect_anomalies(state: ProcessState):
    df = state["current_data"]
    
    anomalies = df[
        (df["mean"] > df["UCL"]) |
        (df["mean"] < df["LCL"]) |
        (df["mean"] > df["upper_1sigma"]) |
        (df["mean"] < df["lower_1sigma"])
    ]
    
    summary = gemini_summarize(
        f"Detected anomalies:\n{anomalies.to_string()}"
    )
    
    state["anomalies"] = anomalies
    state["summaries"]["anomalies"] = summary
    return state

In [37]:
def compare_prev_layer(state: ProcessState):
    curr = state["anomalies"]
    prev = state["prev_data"]
    
    merged = curr.merge(prev, on="wafer_id", suffixes=("_curr", "_prev"))
    
    merged["mean_diff"] = merged["mean_curr"] - merged["mean_prev"]
    merged["sigma_diff"] = merged["sigma_curr"] - merged["sigma_prev"]
    
    threshold = 0.5
    
    merged["from_prev_layer"] = (
        (merged["mean_diff"].abs() < threshold) &
        (merged["sigma_diff"].abs() < threshold)
    )
    
    summary = gemini_summarize(
        f"Comparison with previous layer:\n{merged.to_string()}"
    )
    
    state["comparison_results"] = merged
    state["summaries"]["prev_layer"] = summary
    return state

In [38]:
def check_qual(state: ProcessState):
    df = state["comparison_results"]
    qual = state["qual_data"]
    
    merged = df.merge(qual, on="wafer_id", how="left")
    
    def assign_root(row):
        if row["from_prev_layer"]:
            return "Previous Layer Issue"
        elif row["thk_mean"] < row.get("thk_target", row["thk_mean"]):
            return "Low Thickness in Qual"
        else:
            return "Unknown / New Issue"
    
    merged["root_cause"] = merged.apply(assign_root, axis=1)
    
    summary = gemini_summarize(
        f"Qualification analysis:\n{merged.to_string()}"
    )
    
    state["root_cause"] = merged
    state["summaries"]["qual"] = summary
    return state

In [39]:
def generate_report(state: ProcessState):
    df = state["root_cause"]
    
    summary = gemini_summarize(
        f"Generate final report for process analysis:\n{df.to_string()}"
    )
    
    state["summaries"]["final"] = summary
    return state

In [40]:
from langgraph.graph import StateGraph

builder = StateGraph(ProcessState)

builder.add_node("compute_sigma", compute_sigma)
builder.add_node("detect_anomalies", detect_anomalies)
builder.add_node("compare_prev", compare_prev_layer)
builder.add_node("check_qual", check_qual)
builder.add_node("report", generate_report)

builder.set_entry_point("compute_sigma")

builder.add_edge("compute_sigma", "detect_anomalies")
builder.add_edge("detect_anomalies", "compare_prev")
builder.add_edge("compare_prev", "check_qual")
builder.add_edge("check_qual", "report")

graph = builder.compile()

In [41]:
result = graph.invoke({
    "current_data": current_df,
    "prev_data": prev_df,
    "qual_data": qual_df
})

print(result["summaries"]["final"])

TypeError: gemini_summarize() missing 1 required positional argument: 'df'